# TARQL in Google Colab: SELECT, ASK, and CONSTRUCT over CSV/TSV

This notebook installs [TARQL](https://tarql.github.io/), loads a CSV or TSV table from an example, URL, or upload, and allows to perform SPARQL `SELECT` and `CONSTRUCT` queries over this data.

TARQL binds table columns as SPARQL variables. A column called `word` is available as `?word`. The first row in your data must contain the names of the columns.

To run this notebook, hit "Run all" in the Colab (or Jupyter) menu. To run an individual code cell (e.g., a SPARQL query), hit the "Run" triangle for the cell.

## 1. Prepare your system

- Install TARQL (if not already there)
- Configure folder structure for temporary files
- Initialize helper functions.
- Initialize default prefixes (OntoLex, RDF, RDFS, SKOS, OWL)

In Google Colab, you need to run this every single time before starting a session.

In [5]:
#@title Run this cell to initialize your Notepab
from pathlib import Path
import shutil
import subprocess
import urllib.request
import zipfile

TARQL_VERSION = "1.2"
TARQL_DIR = Path(f"/content/tarql-{TARQL_VERSION}")
TARQL_ZIP = Path(f"/content/tarql-{TARQL_VERSION}.zip")
TARQL_URL = f"https://github.com/tarql/tarql/releases/download/v{TARQL_VERSION}/tarql-{TARQL_VERSION}.zip"
TARQL_BIN = TARQL_DIR / "bin" / "tarql"

if shutil.which("java") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jre-headless"], check=True)

if not TARQL_BIN.exists():
    urllib.request.urlretrieve(TARQL_URL, TARQL_ZIP)
    with zipfile.ZipFile(TARQL_ZIP) as archive:
        archive.extractall("/content")
    TARQL_BIN.chmod(0o755)

version = subprocess.run([str(TARQL_BIN), "--version"], text=True, capture_output=True, check=True)
print(version.stdout.strip() or version.stderr.strip())

print()
from pathlib import Path
import atexit
import csv
import html
import os
import re
import shutil
import subprocess
import urllib.request
from urllib.parse import urlparse

import pandas as pd
from IPython.display import HTML, display

WORK_DIR = Path("/content/tarql-work")
RESULT_DIR = WORK_DIR / "results"
WORK_DIR.mkdir(parents=True, exist_ok=True)


def cleanup_temp_files():
    if RESULT_DIR.exists():
        shutil.rmtree(RESULT_DIR)
    RESULT_DIR.mkdir(parents=True, exist_ok=True)


cleanup_temp_files()
atexit.register(cleanup_temp_files)
print(f"Temporary result folder reset: {RESULT_DIR}")

print()
def tarql_command(query_path: Path, ntriples: bool = False):
    cmd = [str(TARQL_BIN)]
    if DELIMITER == "\t":
        cmd.append("--tabs")
    elif DELIMITER != ",":
        cmd.extend(["--delimiter", DELIMITER])
    if ntriples:
        cmd.append("--ntriples")
    cmd.extend([str(query_path), str(TARQL_INPUT_PATH)])
    return cmd


def run_tarql(query_text: str, query_filename: str, ntriples: bool = False) -> str:
    query_path = WORK_DIR / query_filename
    query_path.write_text(query_text.strip() + "\n", encoding="utf-8")
    result = subprocess.run(tarql_command(query_path, ntriples=ntriples), text=True, capture_output=True)
    if result.stderr.strip():
        print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"TARQL failed with exit code {result.returncode}")
    return result.stdout


def split_tarql_row(line: str):
    return [part.strip() for part in line.strip().strip("|").split("|")]


def clean_tarql_cell(value: str) -> str:
    value = value.strip()
    if len(value) >= 2 and value[0] == '"' and value[-1] == '"':
        return value[1:-1]
    return value


def parse_select_table(tarql_output: str) -> pd.DataFrame:
    lines = [line for line in tarql_output.splitlines() if line.strip()]
    table_lines = [line for line in lines if line.lstrip().startswith("|")]
    if not table_lines:
        return pd.DataFrame()
    header = split_tarql_row(table_lines[0])
    rows = [split_tarql_row(line) for line in table_lines[1:]]
    rows = [[clean_tarql_cell(cell) for cell in row] for row in rows]
    return pd.DataFrame(rows, columns=header)


def colab_download_link(path: Path, label: str):
    path = Path(path)
    if str(path).startswith("/content/"):
        href = "/files/" + str(path.relative_to("/content"))
        display(HTML(f'<a href="{html.escape(href)}" target="_blank" download>{html.escape(label)}</a>'))
    else:
        display(HTML(f"Local file: <code>{html.escape(str(path))}</code>"))


def ask_to_select(query_text: str) -> str:
    match = re.search(r"\bASK\b", query_text, flags=re.IGNORECASE)
    if not match:
        raise ValueError("The ASK query must contain the keyword ASK.")
    prefix_part = query_text[:match.start()]
    rest = query_text[match.end():].lstrip()
    if rest.upper().startswith("WHERE"):
        rest = rest[5:].lstrip()
    return prefix_part + "SELECT * WHERE " + rest.rstrip() + " LIMIT 1"


def count_ntriples(text: str) -> int:
    return sum(1 for line in text.splitlines() if line.strip() and not line.lstrip().startswith("#"))

def trigger_colab_download(path: Path):
    path = Path(path)
    try:
        from google.colab import files
    except ImportError:
        print(f"Direct download is a Google Colab feature. Local file: {path}")
        return
    files.download(str(path))

print("Helper functions initialized.")

PREFIXES = {
    "ontolex": "http://www.w3.org/ns/lemon/ontolex#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "lime": "http://www.w3.org/ns/lemon/lime#",
    "vartrans": "http://www.w3.org/ns/lemon/vartrans#",
    "synsem": "http://www.w3.org/ns/lemon/synsem#",
    "decomp": "http://www.w3.org/ns/lemon/decomp#",
    "lexinfo": "http://www.lexinfo.net/ontology/3.0/lexinfo#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "skos": "http://www.w3.org/2004/02/skos/core#",
    "owl": "http://www.w3.org/2002/07/owl#",
}

print("Prefixes initialized: "+", ".join(sorted(PREFIXES)))


import ipywidgets as widgets
from IPython.display import display

sel_textarea = widgets.Textarea(
    value="\n".join([ f"PREFIX {p}: <{n}>" for p,n in sorted(PREFIXES.items())])+
    "\n\nSELECT DISTINCT *\n WHERE {\n\t\n} LIMIT 10",
    placeholder='Type something',
    description='Input:',
    layout=widgets.Layout(width='80%', height='200pt'),
    disabled=False
)

# To access the input later
def on_sel_submit(change):
    sel_value = sel_textarea.value.split("\n")

con_textarea = widgets.Textarea(
    value="\n".join([ f"PREFIX {p}: <{n}>" for p,n in sorted(PREFIXES.items())])+
    "\n\nCONSTRUCT {\n\t# write some triples\n}\nWHERE {\n\t\n}",
    placeholder='Type something',
    description='Input:',
    layout=widgets.Layout(width='80%', height='200pt'),
    disabled=False
)

# To access the input later
def on_con_submit(change):
    con_value = con_textarea.value.split("\n")


print("Text areas initialized")


tarql:      VERSION: 1.2
tarql:      BUILD_DATE: 2019-05-03T18:52:51Z

Temporary result folder reset: /content/tarql-work/results

Helper functions initialized.
Prefixes initialized: decomp, lexinfo, lime, ontolex, owl, rdf, rdfs, skos, synsem, vartrans
Text areas initialized


## 3. Load a CSV or TSV file

Choose `example`, `url`, or `upload`. You need to run this again if you want to upload another dataset, only.

> Note that the first row must contain column names. The header normalization feature is useful for beginners because it turns difficult column names such as `Part of speech` into TARQL variables such as `?Part_of_speech`.

In [6]:
#@title Click to show/hide code
FILE_URL=""
SOURCE_MODE = "example"  # @param ["example", "url", "upload"]
if SOURCE_MODE=="url":
  URL = ""  # @param {type:"string"}
  FILE_URL=URL
FILE_FORMAT = "auto"  # @param ["auto", "csv", "tsv"]
NORMALIZE_COLUMN_NAMES = True  # @param {type:"boolean"}

def detect_delimiter(path: Path, file_format: str = "auto") -> str:
    if file_format == "csv":
        return ","
    if file_format == "tsv":
        return "\t"
    if path.suffix.lower() in {".tsv", ".tab"}:
        return "\t"
    sample = path.read_text(encoding="utf-8", errors="replace")[:8192]
    try:
        return csv.Sniffer().sniff(sample, delimiters=",\t;|").delimiter
    except csv.Error:
        return ","


def safe_var_name(name: str, fallback: str) -> str:
    safe = re.sub(r"[^A-Za-z0-9_]", "_", str(name).strip())
    safe = re.sub(r"_+", "_", safe).strip("_")
    if not safe:
        safe = fallback
    if safe[0].isdigit():
        safe = "c_" + safe
    return safe


def make_unique(names):
    seen = {}
    result = []
    for name in names:
        count = seen.get(name, 0)
        seen[name] = count + 1
        result.append(name if count == 0 else f"{name}_{count + 1}")
    return result


if SOURCE_MODE == "example":
    raw_input_path = WORK_DIR / "example-lexicon.csv"
    raw_input_path.write_text(
        "word,pos,gloss,translation\n"
        "house,noun,building for people,Haus\n"
        "run,verb,move quickly,laufen\n"
        "light,adjective,not heavy,leicht\n",
        encoding="utf-8",
    )
elif SOURCE_MODE == "url":
    if not FILE_URL.strip():
        raise ValueError("Please enter FILE_URL or switch SOURCE_MODE back to example.")
    filename = Path(urlparse(FILE_URL).path).name or "downloaded-table.csv"
    raw_input_path = WORK_DIR / filename
    urllib.request.urlretrieve(FILE_URL, raw_input_path)
elif SOURCE_MODE == "upload":
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload mode requires Google Colab. Use URL mode outside Colab.") from exc
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    filename = next(iter(uploaded))
    raw_input_path = WORK_DIR / filename
    shutil.move(filename, raw_input_path)
else:
    raise ValueError(f"Unsupported SOURCE_MODE: {SOURCE_MODE}")

DELIMITER = detect_delimiter(raw_input_path, FILE_FORMAT)
df = pd.read_csv(raw_input_path, sep=DELIMITER, dtype=str).fillna("")
original_columns = list(df.columns)

if NORMALIZE_COLUMN_NAMES:
    normalized_columns = make_unique([safe_var_name(col, f"col_{i + 1}") for i, col in enumerate(original_columns)])
else:
    normalized_columns = original_columns

df.columns = normalized_columns
working_ext = ".tsv" if DELIMITER == "\t" else ".csv"
TARQL_INPUT_PATH = WORK_DIR / f"tarql-input{working_ext}"
df.to_csv(TARQL_INPUT_PATH, sep=DELIMITER, index=False, encoding="utf-8")

#print(f"Original file: {raw_input_path}")
#print(f"TARQL input file: {TARQL_INPUT_PATH}")
print(f"Detected delimiter: {'TAB' if DELIMITER == chr(9) else repr(DELIMITER)}")
print("\nTARQL variables:")
for original, normalized in zip(original_columns, normalized_columns):
    print(f" ?{normalized}",end="")
    if original != normalized:
        print(f" (from column: {original!r})",end="")
print()

print("\nSample content:")
display(df.head(20))

Detected delimiter: ','

TARQL variables:
 ?word ?pos ?gloss ?translation

Sample content:


,word,pos,gloss,translation
0,house,noun,building for people,Haus
1,run,verb,move quickly,laufen
2,light,adjective,not heavy,leicht


## 5. Submit a SPARQL SELECT query

You can use SPARQL SELECT queries to inspect the content of your file.
This displays the result table, stores it as a temporary CSV file, and exposes a download link.

For the example data, the following query would work

```
SELECT ?word ?pos ?gloss ?translation
WHERE { }
LIMIT 20
```

Also note that you need to define prefixes, but some have been initialized, already (see log message above).

> Note that the `WHERE` block is intentionally empty, because we just return the variables bound for every line. Alternatively, you can put FILTERs there, etc.

In the text area below, you can enter your text. In the next code line, you can run the query.


In [7]:
#@title Write your SELECT query here
if len(normalized_columns)==0:
  raise Exception("Please provide file or URL or select example mode above")
else:
  print("You can use the following variables: ?"+ ", ?".join(normalized_columns))
print()

display(sel_textarea)
sel_textarea.observe(on_sel_submit, names='sel_value')



You can use the following variables: ?word, ?pos, ?gloss, ?translation



Textarea(value='PREFIX decomp: <http://www.w3.org/ns/lemon/decomp#>\nPREFIX lexinfo: <http://www.lexinfo.net/o…

In [8]:
#@title Submit your SELECT query by running this cell

SELECT_QUERY = sel_textarea.value

select_output = run_tarql(SELECT_QUERY, "select-query.sparql")
select_df = parse_select_table(select_output)

if select_df.empty:
    print("No rows returned.")
else:
    display(select_df)

select_result_path = RESULT_DIR / "tarql-select-result.csv"
select_df.to_csv(select_result_path, index=False, encoding="utf-8")
print(f"Rows: {len(select_df):,}")
# print(f"Temporary result file: {select_result_path}")
if SOURCE_MODE != "example":
  trigger_colab_download(select_result_path)

,word,pos,gloss,translation
0,house,noun,building for people,Haus
1,run,verb,move quickly,laufen
2,light,adjective,not heavy,leicht


Rows: 3


## 6. Submit a SPARQL CONSTRUCT query

This creates an RDF file in N-Triples format, prints the number of triples, and exposes a download link.

Note that the original data usually does not contain proper URIs, so you need to create them. You can do that with

- `BIND` (to create a new variable),
- `ENCODE_FOR_URI` (to replace non-standard characters in the local name), - `CONCAT` (to concatenate the namespace URI with the local name), and
- `IRI` (to convert a string to a URI/IRI).

```
PREFIX ex: <http://example.org/lexicon/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
CONSTRUCT {
  ?entry a               ontolex:LexicalEntry ;
         rdfs:label      ?word ;
         ex:partOfSpeech ?pos ;
         ex:gloss        ?gloss ;
         ex:translation  ?translation .
}
WHERE {
  BIND(IRI(CONCAT("http://example.org/lexicon/",
           ENCODE_FOR_URI(STR(?word))))
       AS ?entry)
}
```

In [9]:
#@title Write your CONSTRUCT query here
if len(normalized_columns)==0:
  raise Exception("Please provide file or URL or select example mode above")
else:
  print("You can use the following variables: ?"+ ", ?".join(normalized_columns))
print()

display(con_textarea)
con_textarea.observe(on_con_submit, names='con_value')

You can use the following variables: ?word, ?pos, ?gloss, ?translation



Textarea(value='PREFIX decomp: <http://www.w3.org/ns/lemon/decomp#>\nPREFIX lexinfo: <http://www.lexinfo.net/o…

In [10]:
#@title Submit your CONSTRUCT by running this cell
CONSTRUCT_QUERY = con_textarea.value

construct_output = run_tarql(CONSTRUCT_QUERY, "construct-query.sparql", ntriples=True)
construct_result_path = RESULT_DIR / "tarql-construct-result.nt"
construct_result_path.write_text(construct_output, encoding="utf-8")
triple_count = count_ntriples(construct_output)

print(f"Triples: {triple_count:,}")
# print(f"Temporary result file: {construct_result_path}")
if triple_count > 0:
    line_count=min(triple_count,10)
    print(f"Sample output (first {line_count} lines):")
    with open(construct_result_path, "rt") as output:
      lines=output.read().split("\n")
      for line in lines[:line_count]:
        print(f"\t{line}")
      if triple_count>line_count:
        print("\t...")
if SOURCE_MODE != "example":
  trigger_colab_download(construct_result_path)
  print("check download dialog")


Triples: 0


## 8. Cleanup temporary files

For running in the notebook, TARQL may create temporary files which clutter your directory. Run this to remove them.

In [11]:
#@title Run this cell to cleanup temporary files
cleanup_temp_files()
print(f"Removed temporary result files from: {RESULT_DIR}")

Removed temporary result files from: /content/tarql-work/results
